# MEPS Method Comparison: Euler vs JAX L-BFGS

This notebook reproduces the final scaling plots from the CTMC tutorial notebook,
but computes the MEPS distribution using **both** the original forward-Euler method
(`get_meps()`) and the new JAX softmax L-BFGS method (`get_meps_jax()`).

We compare:
1. **Arrhenius pump generator** — scatter panels of excess EPR vs MEPS EPR at each system size
2. **Cyclic generator** — scaled (σ − σ_MEPS) / σ_MEPS panels
3. **Summary statistics** — how the two methods diverge as systems get harder

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
os.environ['JAX_PLATFORMS'] = 'cpu'
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from ctmc import (
    ContinuousTimeMarkovChain as MC,
    arrhenius_pump_generator,
    cyclic_generator,
    HAS_JAX,
)

assert HAS_JAX, 'JAX is required for this notebook. Install with: pip install jax[cpu] jaxopt'

plt.rcParams.update({'font.size': 10, 'figure.dpi': 120})
print('Setup complete.  HAS_JAX =', HAS_JAX)

## Helper: compute EPR data for both MEPS methods

We build one CTMC per (size, trial) setting, but call `get_meps()` *and*
`get_meps_jax()` on the same rate matrices so the comparison is apples-to-apples.

In [ ]:
import gc

def sweep_both_methods(sizes, trials, generator, **gen_kwargs):
    """Run a size sweep, computing MEPS with both Euler and JAX.

    Returns a dict keyed by str(S), each value a dict with:
        'ness', 'rand', 'unif'       — EPR arrays (N,)
        'meps_euler', 'meps_jax'     — MEPS EPR arrays (N,)
    """
    results = {}
    for S, N_trial in zip(sizes, trials):
        print(f'  S={S:>4d},  N={N_trial:>3d} ...', end=' ', flush=True)

        m = MC(S=S, N=N_trial, generator=generator, **gen_kwargs)
        ness = m.get_ness()

        # Euler MEPS
        meps_euler = m.get_meps(max_iter=5000)
        epr_meps_euler = np.asarray(m.get_epr(meps_euler))

        # JAX MEPS (fresh start from NESS, not from Euler's solution)
        # We delete the cached meps so JAX starts from NESS too
        if hasattr(m, 'meps'):
            delattr(m, 'meps')
        meps_jax = m.get_meps_jax()
        epr_meps_jax = np.asarray(m.get_epr(meps_jax))

        epr_ness = np.asarray(m.get_epr(ness))
        epr_unif = np.asarray(m.get_epr(m.get_uniform()))
        epr_rand = np.asarray(m.get_epr(m.get_random_state()))

        key = str(S)
        results[key] = {
            'ness': epr_ness,
            'rand': epr_rand,
            'unif': epr_unif,
            'meps_euler': epr_meps_euler,
            'meps_jax': epr_meps_jax,
        }

        # Quick summary
        jax_better = np.sum(epr_meps_jax < epr_meps_euler - 1e-12)
        print(f'done.  JAX lower in {jax_better}/{N_trial}')

        # Free memory between sizes
        del m, ness, meps_euler, meps_jax
        gc.collect()

    return results

## 1. Arrhenius pump generator sweep

In [ ]:
pump_sizes  = [3, 9, 27]
pump_trials = [50, 50, 5]

percent  = 0.8
strength = 4

# We need to set n_pumps per size, so we loop manually
pump_results = {}
for S, N_trial in zip(pump_sizes, pump_trials):
    n_pumps = max(1, int(percent * ((S**2 - S) / 2)))
    print(f'  S={S:>4d}, N={N_trial:>3d}, n_pumps={n_pumps} ...', end=' ', flush=True)

    m = MC(S=S, N=N_trial, generator=arrhenius_pump_generator,
           pump_strength=strength, n_pumps=n_pumps)
    ness = m.get_ness()

    meps_euler = m.get_meps(max_iter=5000)
    epr_meps_euler = np.asarray(m.get_epr(meps_euler))

    if hasattr(m, 'meps'):
        delattr(m, 'meps')
    meps_jax = m.get_meps_jax()
    epr_meps_jax = np.asarray(m.get_epr(meps_jax))

    epr_ness = np.asarray(m.get_epr(ness))
    epr_unif = np.asarray(m.get_epr(m.get_uniform()))
    epr_rand = np.asarray(m.get_epr(m.get_random_state()))

    pump_results[str(S)] = {
        'ness': epr_ness, 'rand': epr_rand, 'unif': epr_unif,
        'meps_euler': epr_meps_euler, 'meps_jax': epr_meps_jax,
    }

    jax_better = np.sum(epr_meps_jax < epr_meps_euler - 1e-12)
    print(f'done.  JAX lower in {jax_better}/{N_trial}')

    del m, ness, meps_euler, meps_jax
    import gc; gc.collect()

print('\nPump sweep complete.')

### Scatter panels: EPR − MEPS EPR  (Euler vs JAX, side by side)

In [ ]:
fig, axes = plt.subplots(2, len(pump_sizes), figsize=(3.2 * len(pump_sizes), 8),
                         sharey='row', sharex='col')

state_types = ['ness', 'rand', 'unif']
colors = {'ness': 'C0', 'rand': 'C1', 'unif': 'C2'}

for col, S_str in enumerate(pump_results):
    d = pump_results[S_str]

    for row, (method_key, method_label) in enumerate(
            [('meps_euler', 'Euler'), ('meps_jax', 'JAX L-BFGS')]):
        ax = axes[row, col]
        meps_epr = d[method_key]

        for st in state_types:
            excess = d[st] - meps_epr
            ax.scatter(meps_epr, excess, label=st, alpha=0.2, s=8, c=colors[st])

        ax.set_yscale('log')
        ax.set_ylim(1e-5, 1e2)
        if row == 0:
            ax.set_title(f'S = {S_str}')
        if col == 0:
            ax.set_ylabel(f'{method_label}\nEPR − MEPS EPR')
        if row == 1:
            ax.set_xlabel('MEPS EPR')

axes[0, -1].legend(fontsize=8, loc='upper right')
fig.suptitle('Arrhenius Pump Generator — Euler (top) vs JAX (bottom)', fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

### Direct comparison: where does JAX find a lower EPR?

In [ ]:
fig, axes = plt.subplots(1, len(pump_sizes), figsize=(3.2 * len(pump_sizes), 4),
                         sharey=True)

for col, S_str in enumerate(pump_results):
    ax = axes[col]
    d = pump_results[S_str]
    euler = d['meps_euler']
    jax_  = d['meps_jax']

    # ratio: >1 means Euler found higher EPR (JAX is better)
    ratio = euler / np.maximum(jax_, 1e-30)

    ax.scatter(jax_, ratio, alpha=0.25, s=10, c='C3')
    ax.axhline(1, color='k', ls='--', lw=0.8)
    ax.set_xlabel('JAX MEPS EPR')
    ax.set_title(f'S = {S_str}')
    if col == 0:
        ax.set_ylabel('Euler EPR / JAX EPR')

fig.suptitle('Pump: EPR ratio (>1 means JAX found lower MEPS)', fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

## 2. Cyclic generator sweep

In [ ]:
cyclic_sizes  = [3, 9, 27]
cyclic_trials = [50, 50, 5]

print('Running cyclic sweep...')
cyclic_results = sweep_both_methods(
    cyclic_sizes, cyclic_trials, cyclic_generator
)
print('Cyclic sweep complete.')

### Scaled scatter panels:  (σ − σ_MEPS) / σ_MEPS  (Euler vs JAX)

In [ ]:
fig, axes = plt.subplots(2, len(cyclic_sizes), figsize=(3.2 * len(cyclic_sizes), 8),
                         sharey='row', sharex='col')

for col, S_str in enumerate(cyclic_results):
    d = cyclic_results[S_str]

    for row, (method_key, method_label) in enumerate(
            [('meps_euler', 'Euler'), ('meps_jax', 'JAX L-BFGS')]):
        ax = axes[row, col]
        meps_epr = d[method_key]

        for st in state_types:
            with np.errstate(divide='ignore', invalid='ignore'):
                scaled = (d[st] - meps_epr) / np.maximum(meps_epr, 1e-30)
            ax.scatter(meps_epr, scaled, label=st, alpha=0.15, s=8, c=colors[st])

        ax.set_yscale('log')
        ax.set_xscale('log')
        if row == 0:
            ax.set_title(f'S = {S_str}')
        if col == 0:
            ax.set_ylabel(f'{method_label}\n(σ − σ_MEPS) / σ_MEPS')
        if row == 1:
            ax.set_xlabel('σ_MEPS')

axes[0, -1].legend(fontsize=8, loc='upper right')
fig.suptitle('Cyclic Generator — Euler (top) vs JAX (bottom)', fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

## 3. Summary: mean scaled excess vs system size

For each system size, we compute the mean of (σ_NESS − σ_MEPS) / σ_MEPS
using each MEPS method, and plot against S.

In [ ]:
def summary_curve(results, sizes):
    """Compute mean ± std of scaled NESS excess for both methods."""
    out = {'S': sizes, 'euler_mean': [], 'euler_std': [],
           'jax_mean': [], 'jax_std': []}
    for S in sizes:
        d = results[str(S)]
        for method, prefix in [('meps_euler', 'euler'), ('meps_jax', 'jax')]:
            meps_epr = d[method]
            with np.errstate(divide='ignore', invalid='ignore'):
                scaled = (d['ness'] - meps_epr) / np.maximum(meps_epr, 1e-30)
            scaled = scaled[np.isfinite(scaled)]
            out[f'{prefix}_mean'].append(np.mean(scaled))
            out[f'{prefix}_std'].append(np.std(scaled))
    for k in out:
        if k != 'S':
            out[k] = np.array(out[k])
    return out


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# --- Pump ---
sp = summary_curve(pump_results, pump_sizes)
ax1.errorbar(sp['S'], sp['euler_mean'], yerr=sp['euler_std'],
             fmt='o-', capsize=4, label='Euler MEPS', color='C0')
ax1.errorbar(sp['S'], sp['jax_mean'], yerr=sp['jax_std'],
             fmt='s--', capsize=4, label='JAX MEPS', color='C3')

# 1/S reference
S_arr = np.array(pump_sizes, dtype=float)
ref = sp['euler_mean'][0] * S_arr[0] / S_arr
ax1.plot(S_arr, ref, 'k:', lw=1, label='~ 1/S')

ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('System size S')
ax1.set_ylabel('mean (σ_NESS − σ_MEPS) / σ_MEPS')
ax1.set_title('Arrhenius Pump Generator')
ax1.legend(fontsize=9)

# --- Cyclic ---
sc = summary_curve(cyclic_results, cyclic_sizes)
ax2.errorbar(sc['S'], sc['euler_mean'], yerr=sc['euler_std'],
             fmt='o-', capsize=4, label='Euler MEPS', color='C0')
ax2.errorbar(sc['S'], sc['jax_mean'], yerr=sc['jax_std'],
             fmt='s--', capsize=4, label='JAX MEPS', color='C3')

ref2 = sc['euler_mean'][0] * S_arr[0] / S_arr
ax2.plot(S_arr, ref2, 'k:', lw=1, label='~ 1/S')

ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('System size S')
ax2.set_ylabel('mean (σ_NESS − σ_MEPS) / σ_MEPS')
ax2.set_title('Cyclic Generator')
ax2.legend(fontsize=9)

fig.suptitle('Scaled NESS excess vs system size — Euler vs JAX', fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## 4. Per-system EPR comparison histogram

For the largest system size in each sweep, show the distribution of
log₁₀(Euler EPR / JAX EPR). Values > 0 mean JAX found a lower MEPS.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for ax, results, sizes, title in [
    (ax1, pump_results, pump_sizes, 'Pump'),
    (ax2, cyclic_results, cyclic_sizes, 'Cyclic'),
]:
    S_max = str(sizes[-1])
    d = results[S_max]
    ratio = d['meps_euler'] / np.maximum(d['meps_jax'], 1e-30)
    log_ratio = np.log10(np.maximum(ratio, 1e-15))

    ax.hist(log_ratio, bins=40, color='C4', edgecolor='k', alpha=0.7)
    ax.axvline(0, color='k', ls='--', lw=1)
    ax.set_xlabel('log₁₀(Euler EPR / JAX EPR)')
    ax.set_ylabel('Count')
    ax.set_title(f'{title} generator, S = {S_max}')

    pct_jax_better = np.mean(ratio > 1.001) * 100
    pct_same = np.mean(np.abs(ratio - 1) < 0.001) * 100
    ax.text(0.95, 0.90, f'JAX better: {pct_jax_better:.0f}%\nSame: {pct_same:.0f}%',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle('Distribution of MEPS EPR ratio at largest system size', fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

## 5. Hard-case deep dive: strong pump, S = 9

This is the regime where the Euler method is known to get stuck.
We crank up `pump_strength` and compare.

In [ ]:
np.random.seed(42)
S_hard = 9
N_hard = 20
strengths = [2, 4, 8, 12]

fig, axes = plt.subplots(2, len(strengths), figsize=(3.5 * len(strengths), 7),
                         sharey='row')

for col, ps in enumerate(strengths):
    n_pumps = max(1, int(0.8 * ((S_hard**2 - S_hard) / 2)))
    m = MC(S=S_hard, N=N_hard, generator=arrhenius_pump_generator,
           pump_strength=ps, n_pumps=n_pumps)
    m.get_ness()

    meps_euler = m.get_meps(max_iter=10000)
    epr_euler = np.asarray(m.get_epr(meps_euler))

    if hasattr(m, 'meps'):
        delattr(m, 'meps')
    meps_jax = m.get_meps_jax()
    epr_jax = np.asarray(m.get_epr(meps_jax))

    epr_ness = np.asarray(m.get_epr(m.ness))

    for row, (meps_epr, mlabel) in enumerate([
        (epr_euler, 'Euler'), (epr_jax, 'JAX'),
    ]):
        ax = axes[row, col]
        with np.errstate(divide='ignore', invalid='ignore'):
            excess = (epr_ness - meps_epr) / np.maximum(meps_epr, 1e-30)
        ax.scatter(meps_epr, excess, alpha=0.3, s=12, c='C0')
        ax.set_yscale('log')
        if col == 0:
            ax.set_ylabel(f'{mlabel}\n(σ_NESS − σ_MEPS) / σ_MEPS')
        if row == 0:
            ax.set_title(f'pump_strength = {ps}')
        if row == 1:
            ax.set_xlabel('σ_MEPS')

    ratio = epr_euler / np.maximum(epr_jax, 1e-30)
    print(f'strength={ps:>2d}: median ratio={np.median(ratio):.3f}, '
          f'max={ratio.max():.3f}, '
          f'JAX better in {np.sum(ratio > 1.001)}/{N_hard}')

    del m, meps_euler, meps_jax
    import gc; gc.collect()

fig.suptitle(f'Hard case: S={S_hard}, varying pump strength — Euler (top) vs JAX (bottom)',
             fontsize=12, y=1.02)
fig.tight_layout()
plt.show()